In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from models.mnistfe import MNISTFeatureExtractor
import torchvision.transforms as transforms
import torchvision
import torch.utils.data as data
from tqdm import tqdm

EPOCHS = 10
BATCH = 128
device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Device: {device}")

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

train_dataset = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform)
train_loader = data.DataLoader(train_dataset, batch_size=BATCH, shuffle=True, num_workers=4)

test_dataset = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transform)
test_loader = data.DataLoader(test_dataset, batch_size=BATCH, shuffle=False, num_workers=4)
    

classifier = MNISTFeatureExtractor().to(device)
cls_optimizer = optim.Adam(classifier.parameters(), lr=1e-3)
cls_criterion = nn.CrossEntropyLoss()

print("Training CNN Feature Extractor on Real MNIST...")
classifier.train()
for epoch in tqdm(range(EPOCHS)): 
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        cls_optimizer.zero_grad()
        preds, _ = classifier(imgs)
        loss = cls_criterion(preds, labels)
        loss.backward()
        cls_optimizer.step()
classifier.eval()
print("Feature Extractor Trained!\n")

Device: mps
Training CNN Feature Extractor on Real MNIST...


100%|██████████| 10/10 [00:44<00:00,  4.48s/it]

Feature Extractor Trained!



In [5]:
def evaluate(model, test_loader, device):
    model.eval()
    with torch.no_grad():
      accuracies = []
      losses = []

      for x_test, y_test in test_loader:
        x_test = x_test.to(device)
        y_test = y_test.to(device)

        y_pred, _ = model.forward(x_test)
        y_pred = torch.argmax(y_pred, dim=1)

        accuracy = (y_pred == y_test).sum().item()/len(y_test)
        accuracies.append(accuracy)

      return np.mean(accuracies)

acc = evaluate(classifier, test_loader, device)
print(f"Test Acc = {acc*100}")

Test Acc = 99.07041139240506


In [7]:
torch.save(classifier.state_dict(), "stats_models/mnist_classifier.pth")
print("Model saved successfully!")

Model saved successfully!
